# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/acs`. Старые метрики не смешиваются с актуальными.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import minmax_scale
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

In [ ]:
from imblearn.over_sampling import SMOTE

# Считываем исходные данные пациентов с CRT

In [ ]:
crt_1 = pd.read_excel('db_crt_1.xlsx', sheet_name='База 1')

In [ ]:
rename_dict ={
    #"Gender": "Is female", 
    "Age": "Age",
    "NYHA class Initial metres": "NYHA class",
    "EF Simp init": "EF LV",
    "Ischemic origin": "Ischemic origin",
    "CRT therapy": "CRT therapy", 
    "Degree of obesity": "Degree of obesity",
    "1год смерть": "Is dead"
}

In [ ]:
#crt_1['Gender'] = crt_1['Gender'] - 1

# Проводилась ли CRT-терапия
crt_1['CRT therapy'] = 1

# Степень ожирения
bins = [0, 30, 35, 40, float('inf')]
labels = [0, 1, 2, 3]
crt_1['Degree of obesity'] = pd.cut(
    crt_1['ИМТ '],
    bins=bins,
    labels=labels,
    right=False,  # Учитываем левую границу интервала (например, 30 включается в 1-ю степень)
    include_lowest=True
).astype(int)

# Ишимическое происхождение 
crt_1['Ischemic origin'] = ((crt_1['AH'] == 1) | (crt_1['AH+CHD'] == 1)).astype(int)

In [ ]:
patients_1 = crt_1[rename_dict.keys()].rename(columns=rename_dict)
patients_1

In [ ]:
patients_1['Degree of obesity'].unique()

In [ ]:
patients_1['Ischemic origin'].unique()

In [ ]:
patients_1['NYHA class'].unique()

# Считываем исходные данные пациентов без CRT

In [ ]:
crt_2 = pd.read_excel('db_crt_3.xlsx')

In [ ]:
rename_dict ={
    #"sex (1-мужчины; 2-женщины)": "Is female", 
    "age": "Age",
    "ФК(NYHA)": "NYHA class",
    "ФВ(Simpson)": "EF LV",
    "СД": "Ischemic origin",
    "CRT therapy": "CRT therapy", 
    "ожирение степнь": "Degree of obesity",
    "исходы 0-жив 1-смерть ": "Is dead"
}

In [ ]:
# Проводилась ли CRT-терапия
crt_2['CRT therapy'] = 0

In [ ]:
patients_2 = crt_2[rename_dict.keys()].rename(columns=rename_dict)
patients_2

In [ ]:
patients_2['Age'] = patients_2['Age'].str.replace(',', '.').astype(float).astype(int)
patients_2['EF LV'] = patients_2['EF LV'].str.replace(',', '.').astype(float)
patients_2['Ischemic origin'] = patients_2['Ischemic origin'].str.replace(',', '.').astype(float).astype(int)
patients_2['Ischemic origin'] = patients_2['Ischemic origin'].replace({1: 0, 2: 1})

In [ ]:
patients_2

# Объеденяем таблицы (те кто получил CRT-терапию и те, кто без CRT терапии)

In [ ]:
patients_crt = pd.concat([patients_1, patients_2]).reset_index(drop=True)
patients_crt.info()

In [ ]:
patients_crt['NYHA class'].unique()

In [ ]:
patients_crt['Ischemic origin'].unique()

In [ ]:
patients_crt['CRT therapy'].unique()

In [ ]:
patients_crt['Degree of obesity'].unique()

In [ ]:
patients_crt['Is dead'].unique()

In [ ]:
patients_crt

# Обрабатываем данные

In [ ]:
patients_crt[patients_crt.isna().any(axis=1)]

In [ ]:
patients_crt = patients_crt.dropna().reset_index(drop=True)

In [ ]:
patients_crt.shape

In [ ]:
# Преобразуем категориальные столбцы в one-hot encoding
#one_hot = pd.get_dummies(patients_crt['NYHA class'], prefix='NYHA class').astype(int)
#patients_crt = patients_crt.drop('NYHA class', axis=1).join(one_hot)

#one_hot = pd.get_dummies(patients_crt['Degree of obesity'], prefix='Degree of obesity').astype(int)
#patients_crt = patients_crt.drop('Degree of obesity', axis=1).join(one_hot)

In [ ]:
patients_crt

# Бьём данные на признаки-значения

In [ ]:
X_y = patients_crt.copy()
X = patients_crt.drop(columns=['Is dead'])
y = patients_crt[['Is dead']]

regr_features = ['Age', 'EF LV', 'NYHA class', 'Degree of obesity']

In [ ]:
X_y.describe()

In [ ]:
X_y[X_y['Is dead'] == 0].shape

In [ ]:
86/174

# Нормализуем признаки

In [ ]:
X_transf_norm = X.copy()
X_transf_norm[regr_features] = X[regr_features].apply(minmax_scale)

In [ ]:
X_transf_norm

In [ ]:
X_transf_norm.plot()
plt.show()

## Стандартизуем данные

In [ ]:
scaler = StandardScaler()

X_transf_stand = X.copy()
X_transf_stand[regr_features] = scaler.fit_transform(X[regr_features])

In [ ]:
X_transf_stand.info()

In [ ]:
X_transf_stand.plot()
plt.show()

# Оценка важности признаков

## Метод взаимной информации

In [ ]:
pd.Series(mutual_info_classif(X_transf_norm, y), index=X.columns)

In [ ]:
NTraj = 1000
#arr = np.zeros(X_transf_stand.shape[1])

mutual_info_classif__stand = pd.DataFrame(columns=X_transf_stand.columns)

for i in range(NTraj):
    temp_df = pd.DataFrame(mutual_info_classif(X_transf_stand, y)).T
    temp_df.columns = X_transf_stand.columns

    mutual_info_classif__stand = pd.concat([mutual_info_classif__stand, temp_df])

In [ ]:
mutual_info_classif__stand

In [ ]:
max_features_num = X_transf_stand.shape[1]

importances = mutual_info_classif__stand.mean().values
std = mutual_info_classif__stand.std().values

indices = np.argsort(importances)[::-1]

# Plot the impurity-based feature importances 
plt.figure(figsize=(9,8))
plt.title(f"Важность факторов риска: Метод взаимной информации", fontsize=15)

plt.bar(range(max_features_num), 
        importances[indices][:max_features_num],
        color="gray", 
        yerr=std[indices][:max_features_num], align="center")

plt.xticks(range(max_features_num), 
           X_transf_stand.columns[indices][:max_features_num], 
           rotation=90, 
           fontsize=12)
plt.xlim([-1, max_features_num])
plt.show()

## Метод случайного леса

In [ ]:
y[y['Is dead'] == 0].shape

In [ ]:
y.shape

In [ ]:
#patients_1[patients_1['Is dead'] == 1].shape[0], patients_2[patients_2['Is dead'] == 1].shape[0]

In [ ]:
smote = SMOTE(sampling_strategy={1: 151}, k_neighbors=5, random_state=42)  # Увеличиваем класс "умер" до 151 образцов

In [ ]:
X_res, y_res = smote.fit_resample(X, y)

In [ ]:
pd.concat([X_res, y_res], axis=1)

In [ ]:
pd.concat([X_res, y_res], axis=1)[174:].sample(10)

In [ ]:
X_res_transf_norm = X_res.copy()
X_res_transf_norm[regr_features] = X_res[regr_features].apply(minmax_scale)

In [ ]:
max_features_num = X_res_transf_norm.shape[1]

# TODO: X_transf_norm
train_x = X_res_transf_norm
train_y = y_res

forest = RandomForestClassifier(n_estimators=1000)
forest.fit(X=train_x, y=train_y)    

# это среднее, т.е. аналогично np.mean([tree.feature_importances_ for tree in forest.estimators_], axis=0)
importances = forest.feature_importances_
# вычисляем стандартное отклонение
std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)
# получаем индексы признаков отсортированных от большего к меньшему: [2, 5, 9, 7, 3, 1, 8, 4, 6, 0]
indices = np.argsort(importances)[::-1]

# Plot the impurity-based feature importances of the forest
plt.figure(figsize=(9,8))
#plt.title(f"Важность показателей: Метод случайного леса", fontsize=15)

plt.bar(range(max_features_num), 
        importances[indices][:max_features_num],
        color="r", 
        yerr=std[indices][:max_features_num], align="center")

plt.xticks(range(max_features_num), 
           train_x.columns[indices][:max_features_num], 
           rotation=90, 
           fontsize=14)
plt.yticks(fontsize=14)
plt.xlim([-1, max_features_num])

plt.savefig('random_forest_smote.png', dpi=700, bbox_inches = "tight")

plt.show()
#random_forest_important__stand = list(train_x.columns[indices][:max_features_num])

In [ ]:
X_res_transf_norm.shape[1]

In [ ]:
max_features_num = X_res_transf_norm.shape[1]

# TODO: X_transf_norm
train_x = X_res_transf_norm
train_y = y_res

forest = RandomForestClassifier(n_estimators=1000)
forest.fit(X=train_x, y=train_y)    

# это среднее, т.е. аналогично np.mean([tree.feature_importances_ for tree in forest.estimators_], axis=0)
importances = forest.feature_importances_
# вычисляем стандартное отклонение
std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)
# получаем индексы признаков отсортированных от большего к меньшему: [2, 5, 9, 7, 3, 1, 8, 4, 6, 0]
indices = np.argsort(importances)[::-1]

# Plot the impurity-based feature importances of the forest
plt.figure(figsize=(9,8))
#plt.title(f"Важность показателей: Метод случайного леса", fontsize=15)

plt.bar(range(max_features_num), 
        importances[indices][:max_features_num],
        color="r", 
        yerr=std[indices][:max_features_num], align="center")

plt.xticks(range(max_features_num), 
           train_x.columns[indices][:max_features_num], 
           rotation=90, 
           fontsize=14)
plt.yticks(fontsize=14)
plt.xlim([-1, max_features_num])

plt.savefig('random_forest_smote.png', dpi=700, bbox_inches = "tight")

plt.show()
#random_forest_important__stand = list(train_x.columns[indices][:max_features_num])

In [ ]:
max_features_num = X_res_transf_norm.shape[1]

# TODO: X_transf_norm
train_x = X_res_transf_norm
train_y = y_res

forest = RandomForestClassifier(n_estimators=1000)
forest.fit(X=train_x, y=train_y)    

# это среднее, т.е. аналогично np.mean([tree.feature_importances_ for tree in forest.estimators_], axis=0)
importances = forest.feature_importances_
# вычисляем стандартное отклонение
std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)
# получаем индексы признаков отсортированных от большего к меньшему: [2, 5, 9, 7, 3, 1, 8, 4, 6, 0]
indices = np.argsort(importances)[::-1]

# Plot the impurity-based feature importances of the forest
plt.figure(figsize=(9,8))
#plt.title(f"Важность показателей: Метод случайного леса", fontsize=15)

plt.bar(range(max_features_num), 
        importances[indices][:max_features_num],
        color="r", 
        yerr=std[indices][:max_features_num], align="center")

plt.xticks(range(max_features_num), 
           train_x.columns[indices][:max_features_num], 
           rotation=90, 
           fontsize=14)
plt.yticks(fontsize=14)
plt.xlim([-1, max_features_num])

plt.savefig('random_forest_smote2.png', dpi=700, bbox_inches = "tight")

plt.show()
#random_forest_important__stand = list(train_x.columns[indices][:max_features_num])

In [ ]:
#np.array([tree.feature_importances_ for tree in forest.estimators_]).shape

In [ ]:
#np.mean(np.array([tree.feature_importances_ for tree in forest.estimators_]), axis=0)

In [ ]:
#forest.feature_importances_

In [ ]:
#np.argsort(forest.feature_importances_)[::-1]

In [ ]:
#indices.shape

In [ ]:
max_features_num = X_transf_norm.shape[1]

# TODO: X_transf_norm
train_x = X_transf_norm
train_y = y

forest = RandomForestClassifier(n_estimators=1000)
forest.fit(X=train_x, y=train_y)    

# это среднее, т.е. аналогично np.mean([tree.feature_importances_ for tree in forest.estimators_], axis=0)
importances = forest.feature_importances_
# вычисляем стандартное отклонение
std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)
# получаем индексы признаков отсортированных от большего к меньшему: [2, 5, 9, 7, 3, 1, 8, 4, 6, 0]
indices = np.argsort(importances)[::-1]

# Plot the impurity-based feature importances of the forest
plt.figure(figsize=(9,8))
#plt.title(f"Важность показателей: Метод случайного леса", fontsize=15)

plt.bar(range(max_features_num), 
        importances[indices][:max_features_num],
        color="r", 
        yerr=std[indices][:max_features_num], align="center")

plt.xticks(range(max_features_num), 
           train_x.columns[indices][:max_features_num], 
           rotation=90, 
           fontsize=14)
plt.yticks(fontsize=14)
plt.xlim([-1, max_features_num])

plt.savefig('random_forest.png', dpi=700, bbox_inches = "tight")

plt.show()
#random_forest_important__stand = list(train_x.columns[indices][:max_features_num])

In [ ]:
confusion_matrix(y_true=y.squeeze().to_numpy(), y_pred=forest.predict(X_transf_norm))

In [ ]:
forest.predict(X_transf_norm)

In [ ]:
forest.predict(X_transf_norm) == y.squeeze().to_numpy()

In [ ]:
# _, ax = plt.subplots(ncols=2, figsize=(6, 4), sharey=True, constrained_layout=True)

# features_info["centered"] = False

# display = PartialDependenceDisplay.from_estimator(
#     hgbdt_model_without_interactions,
#     X_train,
#     **features_info,
#     ax=ax,
#     **common_params,
# )
# _ = display.figure_.suptitle("ICE and PDP representations", fontsize=16)

In [ ]:
X_transf_norm

# Модели машинного обучения

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
X_transf_stand.tail(3)

In [ ]:
y['Is dead'].squeeze().unique()

## Логистическая регрессия

In [ ]:
#X_train, X_test, y_train, y_test = train_test_split(X_transf_stand, y.squeeze(), test_size=0.2, random_state=42)

In [ ]:
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_transf_stand, y.squeeze())

In [ ]:
# Оценка точности
#y_pred = model.predict_proba(X_transf_stand)
y_pred = model.predict(X_transf_stand)
print(f"Accuracy: {accuracy_score(y, y_pred):.2f}")

In [ ]:
y_pred

In [ ]:
y.squeeze().values

In [ ]:
import shap

In [ ]:
explainer = shap.KernelExplainer(model.predict_proba, X_transf_stand)
shap_values = explainer(X_transf_stand)

In [ ]:
#shap_values = explainer.shap_values(X_transf_stand)

In [ ]:
X.columns

In [ ]:
shap.plots.bar(shap_values[:,:,1], show=False)
plt.savefig("shap_logregr.jpg", dpi=700, bbox_inches = "tight")
plt.show() 

In [ ]:
shap_values[:,:,1].shape

In [ ]:
shap.summary_plot(shap_values[:,:,1], X_transf_stand, feature_names=X.columns, plot_type='bar')

## Метод Random Forest

In [ ]:
X_res_transf_norm

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_res_transf_norm, y_res.squeeze(), test_size=0.2, random_state=42)

In [ ]:
# Лучшие параметры: {'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
model = RandomForestClassifier(
    n_estimators=50,
    max_depth=7,          # Ограничиваем глубину против переобучения
    min_samples_split=2,  # Минимум 5 samples для разделения узла
    min_samples_leaf=1,
    random_state=42
)
model.fit(X_train, y_train.squeeze())
#model.fit(X_res_transf_norm, y_res.squeeze())

In [ ]:
y_pred_train = model.predict(X_train) #X_transf_stand)
#y_pred = model.predict_proba(X_transf_stand)
print(f"Accuracy: {accuracy_score(y_train, y_pred_train):.2f}")

In [ ]:
y_pred_train

In [ ]:
y_train.squeeze().values

In [ ]:
y_pred = model.predict(X_test) #X_transf_stand)
#y_pred = model.predict_proba(X_transf_stand)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

In [ ]:
y_pred

In [ ]:
y_test.squeeze().values

In [ ]:
model.predict_proba(X_test)

In [ ]:
confusion_matrix(y_true=y_test.squeeze().values, y_pred=y_pred)

### Оценка гиперпараметров

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150, 200],        # Число деревьев
    'max_depth': [3, 5, 7, 10, None],      # Глубина (None для неограниченной)
    'min_samples_split': [2, 5, 7, 10],    # Минимальное число samples для разделения узла
    'min_samples_leaf': [1, 2, 4]          # Минимальное число samples в листе
}

In [ ]:
model = RandomForestClassifier(random_state=42)

# Для медицинских данных используйте scoring='f1_macro' или 'recall'
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring='recall',    # Или 'recall' если важнее не пропустить умерших
    n_jobs=-1,             # Использовать все ядра CPU
    verbose=1
)

grid_search.fit(X_res_transf_norm, y_res.squeeze())

In [ ]:
print("Лучшие параметры:", grid_search.best_params_)
print("Лучший score (recall):", grid_search.best_score_)

# Лучшая модель
best_rf = grid_search.best_estimator_

In [ ]:
best_rf

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(X_res_transf_norm.columns, best_rf.feature_importances_)
plt.title('Важность признаков')
plt.show()

In [ ]:
plt.savefig('random_forest_smote.png', dpi=700, bbox_inches = "tight")

In [ ]:
# Лучшие параметры: {'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
best_rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=7,          # Ограничиваем глубину против переобучения
    min_samples_split=2,  # Минимум 5 samples для разделения узла
    min_samples_leaf=1,
    random_state=42
)
best_rf.fit(X_res_transf_norm, y_res.squeeze())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Допустим, у вас уже есть обученная модель
# model = RandomForestClassifier().fit(X, y) 

# Получаем важность признаков и их названия
importances = best_rf.feature_importances_
feature_names = X_res_transf_norm.columns

# Сортируем признаки по важности
sorted_idx = np.argsort(importances)
sorted_features = [feature_names[i] for i in sorted_idx]
sorted_importances = [importances[i] for i in sorted_idx]

# Создаем горизонтальный bar plot
plt.figure(figsize=(12, 6))
plt.barh(sorted_features, sorted_importances)
plt.xlabel('Feature Importance', fontsize=14)
#plt.ylabel('Features', fontsize=12)
#plt.title('Features importance', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Добавляем значения важности на график
for i, v in enumerate(sorted_importances):
    plt.text(v + 0.005, i, f"{v:.3f}", color='black', va='center')

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlim([0, 0.3])

plt.tight_layout()
plt.savefig('random_forest_smote.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
max_features_num = X_res_transf_norm.shape[1]

# это среднее, т.е. аналогично np.mean([tree.feature_importances_ for tree in forest.estimators_], axis=0)
importances = best_rf.feature_importances_
# вычисляем стандартное отклонение
std = np.std([tree.feature_importances_ for tree in best_rf.estimators_], axis=0)
# получаем индексы признаков отсортированных от большего к меньшему: [2, 5, 9, 7, 3, 1, 8, 4, 6, 0]
indices = np.argsort(importances)[::-1]

# Plot the impurity-based feature importances of the forest
plt.figure(figsize=(9,8))
#plt.title(f"Важность показателей: Метод случайного леса", fontsize=15)

plt.bar(range(max_features_num), 
        importances[indices][:max_features_num],
        color="r", 
        yerr=std[indices][:max_features_num], align="center")

plt.xticks(range(max_features_num), 
           X_res_transf_norm.columns[indices][:max_features_num], 
           rotation=90, 
           fontsize=14)
plt.yticks(fontsize=14)
plt.xlim([-1, max_features_num])

plt.savefig('random_forest_best_rf_smote.png', dpi=700, bbox_inches = "tight")

plt.show()

# Метод SHAP

In [ ]:
import shap
sns.set_style('whitegrid')

In [ ]:
explainer = shap.TreeExplainer(model)

In [ ]:
shap_values = explainer(X_res_transf_norm)

In [ ]:
shap_values.shape

In [ ]:
X_y.head()

In [ ]:
X_y[X_y['Is dead'] == 1].sort_values('Age')

In [ ]:
#X_y[(X_y['NYHA Initial metres'] < 200)&(X_y['EF Simp init'] < 37)]

## Оценка умерших по каждому признаку

In [ ]:
X_y

In [ ]:
# pd.concat([X_res, y_res], axis=1)

X_y_nyha = X_y.sort_values('NYHA class')[['NYHA class', 'Is dead']].reset_index(drop=True)
idx_nyha_dead = X_y_nyha[X_y_nyha['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_nyha['NYHA class'].plot(label='All patients')
plt.scatter(idx_nyha_dead, X_y_nyha['NYHA class'].iloc[idx_nyha_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Тест шестиминутной ходьбы', fontsize=15)
plt.xlabel('Patients #', fontsize=14)
plt.ylabel('NYHA class', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'nyha_class.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
X_y

In [ ]:
X_y_imt = X_y.sort_values('BMI')[['BMI', 'Is dead']].reset_index(drop=True)
idx_imt_dead = X_y_imt[X_y_imt['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_imt['BMI'].plot(label='All patients')
plt.scatter(idx_imt_dead, X_y_imt['BMI'].iloc[idx_imt_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Индекс массы тела', fontsize=15)
plt.xlabel('Patients #', fontsize=14)
plt.ylabel('Body Mass Index (BMI)', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'bmi.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
X_y.shape

In [ ]:
X_y_age = X_y.sort_values('Age')[['Age', 'Is dead']].reset_index(drop=True)
idx_age_dead = X_y_age[X_y_age['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_age['Age'].plot(label='All patients')
plt.scatter(idx_age_dead, X_y_age['Age'].iloc[idx_age_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Age of the patient', fontsize=15)
plt.xlabel('Patient #', fontsize=14)
plt.ylabel('Age of the patient', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'age.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
X_y_ef = X_y.sort_values('EF LV')[['EF LV', 'Is dead']].reset_index(drop=True)
idx_ef_dead = X_y_ef[X_y_ef['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_ef['EF LV'].plot(label='All patients')
plt.scatter(idx_ef_dead, X_y_ef['EF LV'].iloc[idx_ef_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Фракция выброса левого желудочка', fontsize=15)
plt.xlabel('Patient #', fontsize=14)
plt.ylabel('Left Ventricular Ejection Fraction', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'EF_simp_init.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
X_y

In [ ]:
X_y_io = X_y.sort_values('Ischemic origin')[['Ischemic origin', 'Is dead']].reset_index(drop=True)
idx_ef_dead = X_y_io[X_y_io['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_io['Ischemic origin'].plot(label='All patients')
#plt.scatter(X_y_io.index, X_y_io['Ischemic origin'], marker='o', color='blue', label='All patients')

plt.scatter(idx_ef_dead, X_y_io['Ischemic origin'].iloc[idx_ef_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Фракция выброса левого желудочка', fontsize=15)
plt.xlabel('Patient #', fontsize=14)
plt.ylabel('Ischemic origin', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'ischemic_origin.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
X_y_crt = X_y.sort_values('CRT therapy')[['CRT therapy', 'Is dead']].reset_index(drop=True)
idx_ef_dead = X_y_crt[X_y_crt['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_crt['CRT therapy'].plot(label='All patients')
plt.scatter(idx_ef_dead, X_y_crt['CRT therapy'].iloc[idx_ef_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Фракция выброса левого желудочка', fontsize=15)
plt.xlabel('Patient #', fontsize=14)
plt.ylabel('CRT therapy', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'CRT_therapy.png', dpi=700, bbox_inches = "tight")
plt.show()

## Визуализация SHAP-значений для отдельных образцов

In [ ]:
idxs = X_y[X_y['Is dead'] == 1].index

In [ ]:
len(idxs)

In [ ]:
idxs

In [ ]:
# X_y['Is female'] = X_y['Is female'].astype(int)
# X_y['Is dead']= X_y['Is dead'].astype(int)

# X_y['Age'] = X_y['Age'].astype(int)
# X_y['FC CHF'] = X_y['FC CHF'].astype(int)
# X_y['QRS'] = X_y['QRS'].astype(int)
# X_y['NYHA initial metres'] = X_y['NYHA initial metres'].astype(int)

In [ ]:
X_y.loc[idxs]

In [ ]:
#X_transf_norm['Is female'] = X_transf_norm['Is female'].astype(int)

In [ ]:
X_transf_norm.loc[idxs].shape

In [ ]:
#X_y[X_y.is_dead == 1].sort_values('Age')
X_y['NYHA initial metres'].mean(), X_y['EF LV'].mean(), X_y['BMI'].mean()

In [ ]:
X_y.describe()

In [ ]:
# E[f(X)] = 0.219 -- средний прогноз по всем значениям
# f(x) - прогноз по конкретному образцу x из датасета X_transf_norm
# цветные цифры-подписи - это SHAP-значение каждого признака. Если сложить их с E[f(X)] то получим f(x)
# Чем больше абсолютное значение SHAP-значения признака, тем больше вклад признака в прогноз

# визуализация SHAP-значений для 1, 2 и 3-го образцов
for i in idxs:#[76, 66, 59, 65]: #range(3):
    print(X_y.loc[i])
    shap.plots.waterfall(shap_values[i,:,1],show=False)
    plt.savefig(f'patient_{i}', dpi=700, bbox_inches = "tight")
    plt.show()

In [ ]:
# визуализация SHAP-значений для 1, 2 и 3-го образцов
for i in range(3):
    shap.plots.force(shap_values[i,:,1])
    plt.show()

In [ ]:
shap.plots.force(shap_values[i,:,1])

## Усреднение абсолютных SHAP-значений

In [ ]:
shap.plots.bar(shap_values[:,:,1], show=False)
plt.savefig("shap.jpg", dpi=700, bbox_inches = "tight")
plt.show() 

In [ ]:
X['BMI'].median()

In [ ]:
fig = plt.figure() 
shap.plots.beeswarm(shap_values[:,:,1])
fig.savefig("shap_beeswarm.png", dpi=700, bbox_inches = "tight")  
#plt.savefig("shap_beeswarm.png", dpi=700, bbox_inches = "tight")
plt.close(fig)
plt.show() 

In [ ]:
for col in X.columns:#["NYHA Initial metres", "ИМТ", "EF Simp init", "is_female"]:
    shap.plots.scatter(shap_values[:,col,1], color=shap_values[:,:,1])

# Метод SHAP (SMOTE)

In [ ]:
import shap
sns.set_style('whitegrid')

In [ ]:
explainer = shap.TreeExplainer(best_rf)

In [ ]:
X_y_res = pd.concat([X_res, y_res], axis=1)

In [ ]:
X_y[X_y['Is dead'] == 1]

In [ ]:
X_y[X_y['EF LV'] < 24].shape

In [ ]:
shap_values = explainer(X_transf_norm)

In [ ]:
X_res_transf_norm.plot()
plt.show()

## Оценка умерших по каждому признаку

In [ ]:
list(rename_dict.values())[:-1]

In [ ]:
for feature in list(rename_dict.values())[:-1]:
    # Создаем фигуру с двумя subplots (1 строка, 2 столбца)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    
    # Первый график (слева)
    X_y_feat = X_y.sort_values(feature)[[feature, 'Is dead']].reset_index(drop=True)
    idx_feat_dead = X_y_feat[X_y_feat['Is dead'] == 1].index
    X_y_feat[feature].plot(ax=ax1, label='All patients')
    ax1.scatter(idx_feat_dead, X_y_feat[feature].iloc[idx_feat_dead], marker='o', color='red', label='Deceased patients')
    ax1.set_xlabel('Patient #', fontsize=14)
    ax1.set_ylabel(feature, fontsize=14)
    ax1.tick_params(axis='both', which='major', labelsize=12)
    ax1.legend(fontsize=12)
    
    # Второй график (справа)
    X_y_feat = X_y_res.sort_values(feature)[[feature, 'Is dead']].reset_index(drop=True)
    idx_feat_dead = X_y_feat[X_y_feat['Is dead'] == 1].index
    X_y_feat[feature].plot(ax=ax2, label='All patients')
    ax2.scatter(idx_feat_dead, X_y_feat[feature].iloc[idx_feat_dead], marker='o', color='red', label='Deceased patients & Generated')
    ax2.set_xlabel('Patient #', fontsize=14)
    ax2.set_ylabel(feature, fontsize=14)
    ax2.tick_params(axis='both', which='major', labelsize=12)
    ax2.legend(fontsize=12)
    
    # Сохраняем и показываем
    plt.tight_layout()
    plt.savefig(f'{feature}_comparison_gen.png', dpi=700, bbox_inches="tight")
    plt.show()

In [ ]:
# X_y_nyha = X_y_res.sort_values('NYHA initial metres')[['NYHA initial metres', 'Is dead']].reset_index(drop=True)
# idx_nyha_dead = X_y_nyha[X_y_nyha['Is dead'] == 1].index

# plt.figure(figsize=(9, 6))

# X_y_nyha['NYHA initial metres'].plot(label='All patients')
# plt.scatter(idx_nyha_dead, X_y_nyha['NYHA initial metres'].iloc[idx_nyha_dead], marker='o', color='red', label='Deceased patients')

# #plt.title('Тест шестиминутной ходьбы', fontsize=15)
# plt.xlabel('Patients #', fontsize=14)
# plt.ylabel('NYHA Initial Metres', fontsize=14)
# plt.tick_params(axis='both', which='major', labelsize=12)

# plt.legend(fontsize=12)
# plt.savefig(f'nyha_initial_metres_smote.png', dpi=700, bbox_inches = "tight")
# plt.show()

In [ ]:
X_y_ef = X_y_res.sort_values('EF LV')[['EF LV', 'Is dead']].reset_index(drop=True)
idx_ef_dead = X_y_ef[X_y_ef['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_ef['EF LV'].plot(label='All patients')
plt.scatter(idx_ef_dead, X_y_ef['EF LV'].iloc[idx_ef_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Фракция выброса левого желудочка', fontsize=15)
plt.xlabel('Patient #', fontsize=14)
plt.ylabel('Left Ventricular Ejection Fraction', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'EF_simp_init_smote.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
X_y_ef = X_y_res.sort_values('CRT therapy')[['CRT therapy', 'Is dead']].reset_index(drop=True)
idx_ef_dead = X_y_ef[X_y_ef['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_ef['CRT therapy'].plot(label='All patients')
plt.scatter(idx_ef_dead, X_y_ef['QRS'].iloc[idx_ef_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Фракция выброса левого желудочка', fontsize=15)
plt.xlabel('Patient #', fontsize=14)
plt.ylabel('QRS', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'QRS_smote.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
X_y_ef = X_y_res.sort_values('CSR')[['CSR', 'Is dead']].reset_index(drop=True)
idx_ef_dead = X_y_ef[X_y_ef['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_ef['CSR'].plot(label='All patients')
plt.scatter(idx_ef_dead, X_y_ef['CSR'].iloc[idx_ef_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Фракция выброса левого желудочка', fontsize=15)
plt.xlabel('Patient #', fontsize=14)
plt.ylabel('CSR', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'CSR_smote.png', dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
X_y_ef = X_y_res.sort_values('рСДЛА')[['рСДЛА', 'Is dead']].reset_index(drop=True)
idx_ef_dead = X_y_ef[X_y_ef['Is dead'] == 1].index

plt.figure(figsize=(9, 6))

X_y_ef['рСДЛА'].plot(label='All patients')
plt.scatter(idx_ef_dead, X_y_ef['рСДЛА'].iloc[idx_ef_dead], marker='o', color='red', label='Deceased patients')

#plt.title('Фракция выброса левого желудочка', fontsize=15)
plt.xlabel('Patient #', fontsize=14)
plt.ylabel('рСДЛА', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)

plt.legend(fontsize=12)
plt.savefig(f'рСДЛА_smote.png', dpi=700, bbox_inches = "tight")
plt.show()

## Усреднение абсолютных SHAP-значений

In [ ]:
shap.plots.bar(shap_values[:,:,1], show=False)
plt.savefig("shap.jpg", dpi=700, bbox_inches = "tight")
plt.show()

In [ ]:
fig = plt.figure() 
shap.plots.beeswarm(shap_values[:,:,1])
fig.savefig("shap_beeswarm.png", dpi=700, bbox_inches = "tight")  
#plt.savefig("shap_beeswarm.png", dpi=700, bbox_inches = "tight")
plt.close(fig)
plt.show() 

## Визуализация SHAP-значений для отдельных образцов

In [ ]:
idxs = X_y_res[X_y_res['Is dead'] == 1].index

In [ ]:
X_y.shape

In [ ]:
idxs[idxs < X_y.shape[0]]

In [ ]:
X_y.iloc[idxs[idxs < X_y.shape[0]]]

In [ ]:
for i in idxs[idxs < X_y.shape[0]]:
    print(X_y_res.iloc[i])
    shap.plots.waterfall(shap_values[i,:,1],show=False)
    plt.savefig(f'patient_{i}_smote', dpi=700, bbox_inches = "tight")
    plt.show()

# Кросс-валидация

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, make_scorer, precision_score

In [ ]:
# Разделим данные на обучающую и тестовую выборки (необязательно)
X_train, X_test, y_train, y_test = train_test_split(X_transf_norm, y, test_size=0.2, random_state=42)

# Инициализация модели Random Forest
#model = RandomForestClassifier(random_state=42)
# model = SGDClassifier(loss='log_loss', penalty='elasticnet',
#                             max_iter=1000, tol=1e-3,
#                             random_state=123,
#                             class_weight='balanced')

model = GradientBoostingClassifier()

# Используем F1-score в качестве метрики
f1_scorer = make_scorer(precision_score)

# Выполняем кросс-валидацию
scores = cross_val_score(model, X_train, y_train, cv=3, scoring=f1_scorer)

print("F1-score для каждой итерации кросс-валидации:", scores)
print("Средний F1-score:", scores.mean())

model.fit(X_train, y_train)
precision_score(model.predict(X_test), y_test.to_numpy().squeeze())

In [ ]:
model.predict(X_test)

In [ ]:
y_test.to_numpy().squeeze()

In [ ]:
X_train[:61].shape

In [ ]:
y_train[:61]

In [ ]:
pd.DataFrame(index=X_train[:61].index, columns=['neg', 'pos'],
             data=tf.keras.utils.to_categorical(y_train[:61].copy(), num_classes=2))

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from tensorflow import keras

# Разделим данные на обучающую и тестовую выборки (необязательно)
X_train, X_test, y_train, y_test = train_test_split(X_transf_stand, y, test_size=0.2, random_state=42)

model = SGDClassifier(loss='log_loss', penalty='elasticnet',
                            max_iter=1000, tol=1e-3,
                            random_state=123,
                            class_weight='balanced')

#model = SVC(random_state=42, class_weight='balanced')

#model = RandomForestClassifier(class_weight='balanced', random_state=42)

#model = GradientBoostingClassifier()

# early_stopping = EarlyStopping(monitor='val_accuracy', patience=10)

# model = tf.keras.Sequential()
# model.add(tf.keras.layers.Dense(12, input_dim=X_train.shape[1], activation='relu'))
# #model.add(tf.keras.layers.Dropout(0.3))
# model.add(tf.keras.layers.Dense(2, activation='softmax'))
# model.compile(loss='categorical_crossentropy', 
#               optimizer=tf.keras.optimizers.Adam(learning_rate=1.0e-3), 
#               metrics=['accuracy'])

# tr_y = pd.DataFrame(index=X_train.index, columns=['neg', 'pos'],
#                     data=tf.keras.utils.to_categorical(y_train.copy(), num_classes=2))

# history = model.fit(X_train[:61], 
#                     tr_y[:61], 
#                     epochs=300, 
#                     callbacks=[early_stopping],
#                     batch_size=5, validation_data=(X_train[61:], tr_y[61:]))  

#model = KNeighborsClassifier(n_neighbors=3)

model.fit(X_train[:61], y_train[:61])
model.predict(X_train[61:])

In [ ]:
y_train[61:].to_numpy().squeeze()

In [ ]:
model.predict(X_test)

In [ ]:
y_test.to_numpy().squeeze()

In [ ]:
pd.DataFrame(history.history).loss.plot()
pd.DataFrame(history.history).val_loss.plot()
y_pr = np.argmax(model.predict(X_train[61:]), axis=1)
y_pr

In [ ]:
model.predict(X_train[:61])

In [ ]:
np.argmax(model.predict(X_train[:61]), axis=1)

In [ ]:
y_train[:61].to_numpy().squeeze()

In [ ]:
model.predict(X_train[61:]) == y_train[61:].to_numpy().squeeze()

In [ ]:
model.predict(X_train[:61])

In [ ]:
y_train[:61].to_numpy().squeeze()

In [ ]:
model.predict(X_train[:61]) == y_train[:61].to_numpy().squeeze()